In [5]:
!pip install -q fastapi uvicorn httpx
import os
import sys
import time
import importlib
import subprocess
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression, LogisticRegression
from fastapi.testclient import TestClient

subprocess.run(["pkill", "-f", "uvicorn app:app"], capture_output=True)
subprocess.run(["pkill", "-f", "uvicorn iris_app:app"], capture_output=True)
time.sleep(1)

np.random.seed(42)
n = 500
df = pd.DataFrame()
df["distance_km"] = np.random.uniform(1, 15, n)
df["items"] = np.random.randint(1, 10, n)
df["traffic_level"] = np.random.randint(1, 4, n)
df["delivery_minutes"] = (10 + 3 * df["distance_km"] + 1.5 * df["items"]
                          + 5 * df["traffic_level"] + np.random.normal(0, 3, n))

model = LinearRegression().fit(df[["distance_km", "items", "traffic_level"]], df["delivery_minutes"])
os.makedirs("work", exist_ok=True)
joblib.dump(model, "work/model.joblib")
print("Model saved to work/model.joblib")

app_code = '''import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

model = joblib.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), "model.joblib"))

app = FastAPI(title="Delivery Time Prediction Service")


class Order(BaseModel):
    distance_km: float = Field(gt=0, le=15)
    items: int = Field(ge=1, le=9)
    traffic_level: int = Field(ge=1, le=3)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(order: Order):
    row = pd.DataFrame([order.model_dump()])
    prediction = model.predict(row)[0]
    return {"predicted_minutes": round(float(prediction), 1)}
'''
Path("work/app.py").write_text(app_code, encoding="utf-8")

sys.path.insert(0, "work")
import app
importlib.reload(app)
client = TestClient(app.app)

response = client.get("/health")
print("Health  :", response.status_code, response.json())
response = client.post("/predict", json={"distance_km": 8, "items": 3, "traffic_level": 2})
print("Predict :", response.status_code, response.json())

response = client.post("/predict", json={"distance_km": 8, "items": 3, "traffic_level": 9})
print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])

server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="work",
)

iris = load_iris(as_frame=True)
X_iris = iris.data.copy()
X_iris.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
y_iris = iris.target_names[iris.target]

iris_model = LogisticRegression(max_iter=1000).fit(X_iris, y_iris)
os.makedirs("task", exist_ok=True)
joblib.dump(iris_model, "task/iris_model.joblib")
print("Model saved to task/iris_model.joblib")

iris_code = '''import os
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field

model = joblib.load(os.path.join(os.path.dirname(os.path.abspath(__file__)), "iris_model.joblib"))

app = FastAPI(title="Iris Flower Classification Service")


class Flower(BaseModel):
    sepal_length: float = Field(gt=0, le=10, examples=[5.1])
    sepal_width: float = Field(gt=0, le=10, examples=[3.5])
    petal_length: float = Field(gt=0, le=10, examples=[1.4])
    petal_width: float = Field(gt=0, le=10, examples=[0.2])


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/predict")
def predict(flower: Flower):
    row = pd.DataFrame([flower.model_dump()])
    return {"species": str(model.predict(row)[0])}
'''
Path("task/iris_app.py").write_text(iris_code, encoding="utf-8")

sys.path.insert(0, "task")
import iris_app
importlib.reload(iris_app)
iris_client = TestClient(iris_app.app)

response = iris_client.get("/health")
print("Health  :", response.status_code, response.json())

good_flower = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
response = iris_client.post("/predict", json=good_flower)
print("Predict :", response.status_code, response.json())

bad_flower = {"sepal_length": -5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
response = iris_client.post("/predict", json=bad_flower)
print("Status code:", response.status_code)
print("Problem    :", response.json()["detail"][0]["msg"])
assert response.status_code == 422

iris_server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "iris_app:app", "--host", "0.0.0.0", "--port", "8001"],
    cwd="task",
)
time.sleep(5)
print("Iris server health:", urllib.request.urlopen("http://127.0.0.1:8001/health").read().decode())

try:
    from google.colab.output import eval_js
    print("Delivery docs:", eval_js("google.colab.kernel.proxyPort(8000)") + "docs")
    print("Iris docs    :", eval_js("google.colab.kernel.proxyPort(8001)") + "docs")
except ImportError:
    print("Delivery docs: http://127.0.0.1:8000/docs")
    print("Iris docs    : http://127.0.0.1:8001/docs")

Model saved to work/model.joblib
Health  : 200 {'status': 'ok'}
Predict : 200 {'predicted_minutes': 48.4}
Status code: 422
Problem    : Input should be less than or equal to 3
Model saved to task/iris_model.joblib
Health  : 200 {'status': 'ok'}
Predict : 200 {'species': 'setosa'}
Status code: 422
Problem    : Input should be greater than 0
Iris server health: {"status":"ok"}
Delivery docs: https://8000-m-s-usw3b4-jl7szdmrkki8-b.us-west3.prod.r.colab.devdocs
Iris docs    : https://8001-m-s-usw3b4-jl7szdmrkki8-b.us-west3.prod.r.colab.devdocs
